In [86]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/noc_regions.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/olympics-data.xlsx
/kaggle/input/datasets/lukajincharadze/pandas-data/results.parquet
/kaggle/input/datasets/lukajincharadze/pandas-data/bios.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/coffee.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/results.csv


In [87]:
%pip install -q dagshub mlflow

Note: you may need to restart the kernel to use updated packages.


# DagsHub/Mlflow Initialization

In [88]:
import dagshub
dagshub.init(repo_owner='skoba23', repo_name='ML_Assignment_2', mlflow=True)

Initialized MLflow to track repo "skoba23/ML_Assignment_2"

Repository skoba23/ML_Assignment_2 initialized!

In [89]:
import mlflow
import mlflow.sklearn
import os
TRACKING_URI = "https://dagshub.com/skoba23/ML_Assignment_2.mlflow"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("DecisionTree_Training")

print('MLflow Tracking URI:', TRACKING_URI)
print('Setup complete!')

MLflow Tracking URI: https://dagshub.com/skoba23/ML_Assignment_2.mlflow
Setup complete!


## Data Loading

In [90]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test  = test_transaction.merge(test_identity,   on='TransactionID', how='left')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)
print('Fraud rate: ', round(train['isFraud'].mean() * 100, 2), '%')

Train shape: (590540, 434)
Test shape:  (506691, 433)
Fraud rate:  3.5 %


# Cleaning

In [91]:
mlflow.start_run(run_name = "DT_Cleaning")
null_threshold = 0.5
null_ratio = train.isnull().mean()
#dropping columns
drop_cols = null_ratio[null_ratio > null_threshold].index.tolist()
drop_cols = [c for c in drop_cols if c != "isFraud"]

df_train_clean = train.drop(columns=drop_cols)
df_test_clean  = test.drop(columns=[c for c in drop_cols if c in test.columns])

### Outlier Removal

In [92]:
Q1 = train["TransactionAmt"].quantile(0.25)
Q3 = train["TransactionAmt"].quantile(0.75)
IQR = Q3 - Q1
before = len(df_train_clean)
df_train_clean = df_train_clean[df_train_clean["TransactionAmt"] < Q3 + 3 * IQR]
after = len(df_train_clean)

### Filling null values with median

In [93]:
num_cols = df_train_clean.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c != "isFraud"]
df_train_clean[num_cols] = df_train_clean[num_cols].fillna(df_train_clean[num_cols].median())
df_test_clean[num_cols] = df_test_clean[[c for c in num_cols if c in df_test_clean.columns]].fillna(df_train_clean[[c for c in num_cols if c in df_test_clean.columns]].median())


### One-Hot Encoding

In [94]:
from sklearn.preprocessing import LabelEncoder
cat_cols = df_train_clean.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    df_train_clean[col] = le.fit_transform(df_train_clean[col].astype(str))
    if col in df_test_clean.columns:
        df_test_clean[col] = le.transform(df_test_clean[col].astype(str).map(
            lambda x: x if x in le.classes_ else le.classes_[0]))

In [95]:
mlflow.log_param("null_threshold", null_threshold)
mlflow.log_param("encoding", "LabelEncoder")
mlflow.log_param("dropped_columns", len(drop_cols))
mlflow.log_param("remaining_features", df_train_clean.shape[1])
mlflow.log_param("outlier_method", "IQR_3x")
mlflow.log_metric("removed_outliers", before - after)

print(f"Dropped {len(drop_cols)} columns, remaining: {df_train_clean.shape[1]}")

Dropped 214 columns, remaining: 220


In [96]:
mlflow.end_run()

🏃 View run DT_Cleaning at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1/runs/197c37b8c16344f8a631cbc68b4dff7b
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1


# Feature Engineering

In [97]:
with mlflow.start_run(run_name = "DT_FeatureEngineering"):
    df_train_clean["TransactionAmt_log"] = np.log1p(df_train_clean["TransactionAmt"])
    df_test_clean["TransactionAmt_log"]  = np.log1p(df_test_clean["TransactionAmt"])
    
    df_train_clean["Transaction_day"]  = (df_train_clean["TransactionDT"] // (3600 * 24)) % 7
    df_train_clean["Transaction_hour"] = (df_train_clean["TransactionDT"] // 3600) % 24
    df_test_clean["Transaction_day"]   = (df_test_clean["TransactionDT"] // (3600 * 24)) % 7
    df_test_clean["Transaction_hour"]  = (df_test_clean["TransactionDT"] // 3600) % 24

    train["card1_addr1"] = train["card1"].astype(str) + "_" + train["addr1"].astype(str)

    fraud_rate = train.groupby("card1")["isFraud"].mean()
    train["card1_fraud_rate"] = train["card1"].map(fraud_rate)
    
    mlflow.log_param("new_features", "TransactionAmt_log, Transaction_day, Transaction_hour")
    mlflow.log_metric("total_features", df_train_clean.shape[1])
    
    print("Feature Engineering done: ", df_train_clean.shape)

/tmp/ipykernel_57/2358694610.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_train_clean["TransactionAmt_log"] = np.log1p(df_train_clean["TransactionAmt"])
/tmp/ipykernel_57/2358694610.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_test_clean["TransactionAmt_log"]  = np.log1p(df_test_clean["TransactionAmt"])
/tmp/ipykernel_57/2358694610.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all column

Feature Engineering done:  (554118, 223)
🏃 View run DT_FeatureEngineering at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1/runs/68287d257b124236adaaaf03a4f847ec
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1


# Feature Selection

In [98]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.tree import DecisionTreeClassifier
Y = df_train_clean["isFraud"]
X = df_train_clean.drop(columns=["isFraud", "TransactionID"])

with mlflow.start_run(run_name="DT_FeatureSelection"):
    dt_selector = DecisionTreeClassifier(max_depth=5, random_state=42)
    dt_selector.fit(X, Y)
    
    importances = pd.Series(dt_selector.feature_importances_, index=X.columns)
    selected_features = importances[importances > 0.001].index.tolist()
    
    mlflow.log_param("method", "feature_importances")
    mlflow.log_param("threshold", 0.001)
    mlflow.log_metric("selected_features", len(selected_features))
    print(f"Selected {len(selected_features)} features")

X_final = X[selected_features]
X_test_final = df_test_clean[[c for c in selected_features if c in df_test_clean.columns]]


Selected 20 features
🏃 View run DT_FeatureSelection at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1/runs/e18c46438e6d47f99fcb09376920d4d9
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1


### Filling missing columns with 0

In [99]:
for col in selected_features:
    if col not in X_test_final.columns:
        X_test_final[col] = 0
X_test_final = X_test_final[selected_features]

# Training

In [100]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import mlflow.sklearn

X_tr, X_val, Y_tr, Y_val = train_test_split(X_final, Y, test_size=0.2, random_state=42, stratify=Y)


### Baseline

In [101]:
with mlflow.start_run(run_name="DT_Train_Baseline"):
    pipe1 = Pipeline([
        ("model", DecisionTreeClassifier(random_state=42))
    ])
    pipe1.fit(X_tr, Y_tr)
    preds1 = pipe1.predict_proba(X_val)[:, 1]
    auc1 = roc_auc_score(Y_val, preds1)

    mlflow.log_params({"max_depth": "None", "criterion": "gini"})
    mlflow.log_metric("val_auc", auc1)
    mlflow.sklearn.log_model(pipe1, "model")
    print(f"Baseline AUC: {auc1:.4f}")

2026/05/07 15:39:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 15:39:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Baseline AUC: 0.8000
🏃 View run DT_Train_Baseline at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1/runs/ea5d012669804d96a7253c4d83cf2485
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1


### Max_Depth increase

In [102]:
with mlflow.start_run(run_name="DT_Train_MaxDepth5"):
    pipe2 = Pipeline([
        ("model", DecisionTreeClassifier(max_depth=5, random_state=42))
    ])
    pipe2.fit(X_tr, Y_tr)
    preds2 = pipe2.predict_proba(X_val)[:, 1]
    auc2 = roc_auc_score(Y_val, preds2)

    mlflow.log_params({"max_depth": 5, "criterion": "gini"})
    mlflow.log_metric("val_auc", auc2)
    mlflow.sklearn.log_model(pipe2, "model")
    print(f"MaxDepth=5 AUC: {auc2:.4f}")

2026/05/07 15:40:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 15:40:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MaxDepth=5 AUC: 0.7880
🏃 View run DT_Train_MaxDepth5 at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1/runs/b95e745872f94650a1be8d6aa2a266f3
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1


In [103]:
with mlflow.start_run(run_name="DT_Train_MaxDepth10"):
    pipe3 = Pipeline([
        ("model", DecisionTreeClassifier(max_depth=10, random_state=42))
    ])
    pipe3.fit(X_tr, Y_tr)
    preds3 = pipe3.predict_proba(X_val)[:, 1]
    auc3 = roc_auc_score(Y_val, preds3)

    mlflow.log_params({"max_depth": 10, "criterion": "gini"})
    mlflow.log_metric("val_auc", auc3)
    mlflow.sklearn.log_model(pipe3, "model")
    print(f"MaxDepth=10 AUC: {auc3:.4f}")

2026/05/07 15:40:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 15:40:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MaxDepth=10 AUC: 0.8363
🏃 View run DT_Train_MaxDepth10 at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1/runs/2ea5c43edf6c42be92748f4ea12d4f5f
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1


### Class Weight Balance

In [104]:
with mlflow.start_run(run_name="DT_Train_Balanced"):
    pipe4 = Pipeline([
        ("model", DecisionTreeClassifier(max_depth=10, criterion="entropy",
                                          class_weight="balanced", random_state=42))
    ])
    pipe4.fit(X_tr, Y_tr)
    preds4 = pipe4.predict_proba(X_val)[:, 1]
    auc4 = roc_auc_score(Y_val, preds4)

    mlflow.log_params({"max_depth": 10, "criterion": "entropy", "class_weight": "balanced"})
    mlflow.log_metric("val_auc", auc4)
    mlflow.sklearn.log_model(pipe4, "model")
    print(f"Entropy Balanced AUC: {auc4:.4f}")

2026/05/07 15:40:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 15:40:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Entropy Balanced AUC: 0.8524
🏃 View run DT_Train_Balanced at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1/runs/aede5ded7f444c8c94bf97560bd4244d
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1


### Min_Samples_Leaf 

In [105]:
with mlflow.start_run(run_name="DT_Train_minSamplesLeaf"):
    pipe5 = Pipeline([
        ("model", DecisionTreeClassifier(max_depth=10, criterion="entropy",
                                          min_samples_leaf=50, class_weight="balanced",
                                          random_state=42))
    ])
    pipe5.fit(X_tr, Y_tr)
    preds5 = pipe5.predict_proba(X_val)[:, 1]
    auc5 = roc_auc_score(Y_val, preds5)

    mlflow.log_params({"max_depth": 10, "criterion": "entropy",
                       "min_samples_leaf": 50, "class_weight": "balanced"})
    mlflow.log_metric("val_auc", auc5)
    mlflow.sklearn.log_model(pipe5, "model")
    print(f"minSamplesLeaf AUC: {auc5:.4f}")

2026/05/07 15:40:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 15:40:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


minSamplesLeaf AUC: 0.8595
🏃 View run DT_Train_minSamplesLeaf at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1/runs/c771ffd12806428ba5de46d172abc358
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1


### Cross Validation

In [106]:
best_pipe = max([(auc1, pipe1), (auc2, pipe2), (auc3, pipe3),
                  (auc4, pipe4), (auc5, pipe5)], key=lambda x: x[0])[1]
best_auc  = max(auc1, auc2, auc3, auc4, auc5)

with mlflow.start_run(run_name="DT_CrossValidation"):
    cv_scores = cross_val_score(best_pipe, X_final, Y, cv=5, scoring="roc_auc")

    mlflow.log_param("cv_folds", 5)
    mlflow.log_metric("cv_auc_mean", cv_scores.mean())
    mlflow.log_metric("cv_auc_std", cv_scores.std())
    print(f"CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


CV AUC: 0.8298 ± 0.0176
🏃 View run DT_CrossValidation at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1/runs/f6d51c7d34844cadac5236f28cc0631d
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1


# Saving To Model Registry

In [107]:
with mlflow.start_run(run_name="DT_BestModel_Registry"):
    mlflow.sklearn.log_model(
        best_pipe,
        artifact_path="model",
        registered_model_name="IEEE_Fraud_DecisionTree"
    )
    mlflow.log_metric("val_auc", best_auc)
    print(f"Best model saved! AUC: {best_auc:.4f}")

2026/05/07 15:41:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 15:41:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'IEEE_Fraud_DecisionTree' already exists. Creating a new version of this model...
2026/05/07 15:41:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: IEEE_Fraud_DecisionTree, version 3
Created version '3' of model 'IEEE_Fraud_DecisionTree'.


Best model saved! AUC: 0.8595
🏃 View run DT_BestModel_Registry at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1/runs/a74cf72556ed420f9e98d331bf596262
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/1
